# Real-Time Fraud Scoring on Snowflake — Feature Store + Real-Time Inference

This notebook runs the **whole real-time loop inside Snowflake**: inspect the feature
store, stream live payment events, watch the **Postgres-backed online store** update
within seconds, and score a transaction against a **Real-Time Inference** endpoint —
all over the REST APIs.

### Before you run (one-time, in the notebook's *Service settings*)
1. **Runtime:** Container runtime (e.g. `MLOPS_CPU_M_POOL`).
2. **External access integration:** attach **`RT_FS_DEMO_EAI`**.
3. **Secret:** attach **`FRAUD_RT_DEMO.FEATURE_STORE.DEMO_PAT`**.

The EAI grants egress to the ingest / query / inference hosts and exposes the PAT secret
to the notebook. Without it, the REST calls below are blocked.


## 0. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB, FS = "FRAUD_RT_DEMO", "FEATURE_STORE"
session.sql(f"USE SCHEMA {DB}.{FS}").collect()

ENTITY_KEY    = "ACCOUNT_ID"
FV_VELOCITY   = "ACCOUNT_VELOCITY"
FV_VERSION    = "V1"
STREAM_SOURCE = "TRANSACTION_EVENTS"
MODEL_NAME    = "AML_FRAUD_GBM"
MODEL_VERSION = "V2"
INFER_SERVICE = "AML_FRAUD_RT_SERVICE"
ACCT_HISTORY  = f"{DB}.CURATED.ACCOUNT_HISTORY"

# The Query API returns features as a POSITIONAL array, in this registered order:
VELOCITY_FEATURES = ["TXN_COUNT_1H","TXN_COUNT_24H","AMT_SUM_24H","AMT_SUM_48H",
                     "DISTINCT_BANKS_24H","DISTINCT_RECEIVERS_24H",
                     "CROSS_CCY_CNT_24H","HIGH_RISK_CNT_24H"]

print("role:", session.get_current_role(), "| warehouse:", session.get_current_warehouse())

## 1. Connect to the real-time endpoints

The PAT comes from the attached **secret** (never a literal in code). The three hosts were
resolved once from the online service + SPCS service; if you ever rebuild those services,
re-resolve them (the `demo/` terminal scripts print them) and update the network rule
`RT_FS_DEMO_EAI` accordingly.


In [ ]:
import requests
from snowflake.snowpark.secrets import get_generic_secret_string

PAT = get_generic_secret_string("fraud_rt_demo/feature_store/demo_pat")
HEADERS = {"Authorization": f'Snowflake Token="{PAT}"', "Content-Type": "application/json"}

# Resolved endpoint hosts (must match the RT_FS_DEMO_EAI network rule).
INGEST_URL = "https://mx4lo-sfsenorthamerica-demo156.snowflakecomputing.app"
QUERY_URL  = "https://ay4lo-sfsenorthamerica-demo156.snowflakecomputing.app"
INFER_URL  = "https://iy4lo-sfsenorthamerica-demo156.snowflakecomputing.app"

print("PAT loaded:", bool(PAT), "(len", len(PAT), ")")
print("ingest:", INGEST_URL, "\nquery :", QUERY_URL, "\ninfer :", INFER_URL)

## 2. Show the model\n\nThe trained gradient-boosted model, registered in the Snowflake Model Registry.

In [ ]:
SHOW MODELS IN SCHEMA FRAUD_RT_DEMO.FEATURE_STORE

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=DB, schema_name=FS)
mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)

predict_fn = next(f for f in mv.show_functions() if f["target_method"].lower() == "predict")
SIG = [(str(s.name), str(getattr(s, "_dtype", "")).upper()) for s in predict_fn["signature"].inputs]

print(f"{MODEL_NAME}/{MODEL_VERSION}  -  {len(SIG)} input features")
print("methods:", [f["target_method"] for f in mv.show_functions()])
mv.show_metrics()

## 3. Query online features (REST Query API)

A single low-latency point lookup against the **Postgres online store**. We check an
established account first — quiet history, so the velocity counters are ~zero.


In [ ]:
import pandas as pd

def query_velocity(account):
    r = requests.post(f"{QUERY_URL}/api/v1/query", headers=HEADERS, json={
        "name": FV_VELOCITY, "version": FV_VERSION, "object_type": "feature_view",
        "request_rows": [{"entity": {ENTITY_KEY: account}}]}, timeout=30)
    r.raise_for_status()
    return dict(zip(VELOCITY_FEATURES, r.json()["results"][0]["features"]))

ESTABLISHED = "012719_8019E5AE0"
pd.DataFrame([query_velocity(ESTABLISHED)], index=[ESTABLISHED])

## 4. Stream a fan-out burst (REST Ingest API)

Simulate a brand-new mule account blasting 40 high-value cross-border payments to many
different banks — the classic fan-out laundering pattern.


In [ ]:
import random, time
from datetime import datetime, timedelta

def make_event(account, seq, fraud=True):
    ts = (datetime.utcnow() + timedelta(milliseconds=seq)).strftime("%Y-%m-%d %H:%M:%S.%f")
    return {ENTITY_KEY: account, "EVENT_TS": ts,
            "AMOUNT_PAID": round(random.uniform(50_000, 900_000), 2) if fraud else round(random.uniform(20, 8000), 2),
            "RECEIVER_ACCOUNT_ID": f"{random.randint(1,30000):05d}_{random.randint(0,9_000_000_000):010X}",
            "RECEIVER_BANK": f"{random.randint(1,250000)}",
            "IS_CROSS_CURRENCY": 1 if (fraud and random.random() < 0.7) else 0,
            "IS_HIGH_RISK_FORMAT": 1 if fraud else 0}

MULE = f"NEWMULE_{int(time.time())}"
records = [make_event(MULE, i, fraud=True) for i in range(40)]
# The Ingest API caps records per request, so post in batches of 10.
codes = []
for b in range(0, len(records), 10):
    r = requests.post(f"{INGEST_URL}/api/v1/ingest", headers=HEADERS,
                      json={"dry_run": False, "records": {STREAM_SOURCE: records[b:b+10]}}, timeout=30)
    codes.append(r.status_code)
print("ingest HTTP", codes, "| mule account:", MULE)

## 5. Re-query — features are fresh in seconds

Continuous aggregation updates the online store almost immediately. Same Query API call,
new account: the velocity counters have jumped.


In [ ]:
time.sleep(3)
pd.DataFrame([query_velocity(MULE)], index=[MULE])

## 6. Score the transaction (Real-Time Inference)

Assemble the full feature vector — slow-moving **profile** (from `ACCOUNT_HISTORY`),
real-time **velocity** (Query API), and this transaction's **request context** — align it
to the model signature, and POST to `/predict-proba`.

> If the inference service auto-suspended, run once:
> `ALTER SERVICE FRAUD_RT_DEMO.FEATURE_STORE.AML_FRAUD_RT_SERVICE RESUME;`


In [ ]:
def profile_row(account):
    df = session.sql(f"SELECT * FROM {ACCT_HISTORY} WHERE {ENTITY_KEY}='{account}'").to_pandas()
    return {c.upper(): df[c].iloc[0] for c in df.columns} if len(df) else {}

def coerce(v, dt):
    try:
        f = float(v); f = 0.0 if f != f else f
    except (TypeError, ValueError):
        f = 0.0
    if "BOOL" in dt: return bool(int(f))
    if "INT"  in dt: return int(f)
    return f

def score(account, amount, scenario="fraud", fmt="ACH"):
    feats = {}
    feats.update(profile_row(account))          # profile (empty for a new account)
    feats.update(query_velocity(account))       # real-time velocity
    feats["AMOUNT_PAID"]         = amount
    feats["IS_CROSS_CURRENCY"]   = 1 if scenario == "fraud" else 0
    feats["IS_CROSS_BORDER"]     = 1 if scenario == "fraud" else 0
    feats["IS_HIGH_RISK_FORMAT"] = 1 if fmt in ("Wire", "Bitcoin", "Cash") else 0
    avg = float(feats.get("HIST_AVG_AMOUNT", 0) or 0)
    feats["AMOUNT_TO_AVG_RATIO"] = amount / avg if avg else 1.0
    fmt_col = "PAYMENT_FORMAT_" + fmt.upper().replace(" ", "_")

    dt = dict(SIG)
    row = {n: (coerce(feats[n], t) if n in feats else (0 if ("INT" in t or "BOOL" in t) else 0.0))
           for n, t in SIG}
    if fmt_col in row:
        row[fmt_col] = coerce(1, dt[fmt_col])
    order = [n for n, _ in SIG]
    payload = {"dataframe_split": {"index": [0], "columns": order, "data": [[row[c] for c in order]]}}

    t0 = time.time()
    r = requests.post(f"{INFER_URL}/predict-proba", headers=HEADERS, json=payload, timeout=60)
    ms = (time.time() - t0) * 1000
    proba = r.json()["data"][0][1].get("output_feature_1")
    return proba, ms

p_fraud, ms = score(MULE, 9000, scenario="fraud", fmt="ACH")
print(f">>> NEW-ACCOUNT FAN-OUT  P(fraud) = {p_fraud:.4f}   (round-trip {ms:.0f} ms)")

### Contrast: the same channel, an established account\n\nIdentical ACH channel — but a seasoned profile with quiet velocity scores far lower. That contrast is the proof the score is driven by *new-account + velocity*, not the payment channel alone.

In [ ]:
p_norm, ms2 = score(ESTABLISHED, 2500, scenario="normal", fmt="ACH")
print(f"Established acct, normal ACH:  P(fraud) = {p_norm:.4f}   ({ms2:.0f} ms)")
print(f"New-account fan-out (above):   P(fraud) = {p_fraud:.4f}")